# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata (Croissant Schema)
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")
print(f"Published: {metadata.datePublished}")
print(f"Keywords: {', '.join(metadata.keywords)}")

## 2. Data Overview
Review available record sets (tables), and their field and column `@id`s.

**Note:** Entities in Croissant are referenced by their `@id` field, which is crucial for unambiguous referencing across schema elements.


In [ ]:
# List available RecordSets and their fields using @id
record_sets = [rs for rs in metadata.recordSet]
if not record_sets:
    print("No RecordSet entries are defined directly within the Croissant metadata.")
else:
    for rs in record_sets:
        print(f"RecordSet: {rs['@id']}\n  Name: {rs.get('name')}\n  Description: {rs.get('description')}\n")

    # For demonstration, show fields for the first RecordSet
    first_rs = record_sets[0]
    print("Fields in the first RecordSet:")
    for field in first_rs.get('field', []):
        print(f"  Field @id: {field['@id']} | name: {field.get('name')} | description: {field.get('description')}")

## 3. Data Extraction
Load data from record sets into DataFrames for analysis.

> We use the record set and field `@id` obtained above. If the Croissant schema does not provide direct RecordSet entries, use the main data table RecordSet `@id` from the schema documentation (extracted below as an example; update according to the specific Croissant schema you are working with).


In [ ]:
# If recordSets are not defined in the metadata object, we may need to inspect and extract them from the schema JSON.
# However, the mlcroissant library exposes recordSet ids via dataset.record_sets (with underscore)

# List record set IDs
record_set_ids = dataset.record_sets
print(f"RecordSet @ids found: {record_set_ids}")

# Load records from each record set
dataframes = dict()
for rs_id in record_set_ids:
    recs = list(dataset.records(record_set=rs_id))
    if len(recs):
        dataframes[rs_id] = pd.DataFrame(recs)
        print(f"Loaded RecordSet {rs_id} with {len(recs)} records.")
    else:
        print(f"RecordSet {rs_id} returned zero records.")

# For demonstration, pick the first non-empty record set
if dataframes:
    demo_rs_id = list(dataframes.keys())[0]
    print(f"Demo RecordSet (@id): {demo_rs_id}")
    print(f"Columns: {dataframes[demo_rs_id].columns.tolist()}")
    display(dataframes[demo_rs_id].head())
else:
    print("No records loaded from any RecordSets.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

> You need to select a numeric `@id` (column) available in the chosen record set. Use the columns printed above and replace below accordingly if running this notebook interactively.


In [ ]:
# Example: EDA on a numeric field (e.g., 'Molecular_Weight' or 'Abundance' or another field with numeric values)
import numpy as np

# Demo: Use a field likely to exist, adapt if schema/column headers differ.
if dataframes:
    df = dataframes[demo_rs_id]
    # Try to autodetect likely numeric columns
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"Selected numeric field (as @id/column): {numeric_field}")
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean): {len(filtered_df)} rows")

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head(5))

        # If another column is suitable for grouping (categorical)
        group_field = None
        # Pick a non-numeric column if available
        non_numeric = [col for col in df.columns if col != numeric_field and df[col].dtype==object]
        if non_numeric:
            group_field = non_numeric[0]
            grouped = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"\nGrouped mean {numeric_field} by '{group_field}':")
            print(grouped.head())
    else:
        print("No numeric fields found for EDA.")
else:
    print("No data loaded for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: Histogram for the numeric field, and bar plot by group
if dataframes and 'numeric_field' in locals():
    df = dataframes[demo_rs_id]
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=40, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if 'group_field' in locals() and group_field is not None:
        # Show top categories by count
        sizes = df[group_field].value_counts().head(10)
        plt.figure(figsize=(8,4))
        sns.barplot(x=sizes.values, y=sizes.index)
        plt.title(f"Top 10 by {group_field}")
        plt.xlabel("Count")
        plt.ylabel(group_field)
        plt.show()


## 6. Conclusion
Summarize key findings and observations from the dataset exploration.


* The Croissant schema provides a machine-actionable overview of the extracellular vesicle protein dataset.
* We loaded metadata and explored available record sets and columns by their `@id`.
* Numeric field filtering and basic normalization/grouping were demonstrated; further domain-specific analysis can follow depending on the data field contents.
* The approach using `mlcroissant` provides reproducibility and transparency via schema-driven dataset understanding.
